In [4]:
import pandas as pd 

#use given directory since the csv is too big for github
merged = pd.read_csv("/Users/gianellarobles/Downloads/sample_2013/merged_mortgage_2013.csv")

#all columns for the dataset
merged.columns

/var/folders/63/h0zgkssd48bbn8s4xmhg8hs40000gn/T/ipykernel_93362/487843566.py:4: DtypeWarning: Columns (3,7,23,24,28,56,57,58,59) have mixed types. Specify dtype option on import or set low_memory=False.
  merged = pd.read_csv("/Users/gianellarobles/Downloads/sample_2013/merged_mortgage_2013.csv")


Index(['loan_sequence_number', 'monthly_reporting_period',
       'current_actual_upb', 'current_loan_delinquency_status', 'loan_age',
       'remaining_months_to_legal_maturity', 'defect_settlement_date',
       'modification_flag', 'zero_balance_code', 'zero_balance_effective_date',
       'current_interest_rate', 'current_deferred_upb',
       'due_date_of_last_paid_installment', 'mi_recoveries',
       'net_sales_proceeds', 'non_mi_recoveries', 'expenses', 'legal_costs',
       'maintenance_and_preservation_costs', 'taxes_and_insurance',
       'miscellaneous_expenses', 'actual_loss_calculation',
       'modification_cost', 'step_modification_flag', 'deferred_payment_plan',
       'estimated_loan_to_value', 'zero_balance_removal_upb',
       'delinquent_accrued_interest', 'delinquency_due_to_disaster',
       'borrower_assistance_status_code', 'current_month_modification_cost',
       'interest_bearing_upb', 'credit_score', 'first_payment_date',
       'first_time_homebuyer_flag', 

In [ ]:
#count missing values
missing_count = merged.isnull().sum()

#percentage missing
missing_percent = (merged.isnull().sum() / len(merged)) * 100

#combine into one table
missing_summary = pd.DataFrame({
    "Missing Count": missing_count,
    "Missing Percent": missing_percent
})

#sort from most missing to least
missing_summary = missing_summary.sort_values(
    by="Missing Percent",
    ascending=False
)

#high missing count only
display(missing_summary[missing_summary["Missing Count"] > 0])

,Missing Count,Missing Percent
defect_settlement_date,4132380,99.997871
legal_costs,4132170,99.992789
mi_recoveries,4132170,99.992789
miscellaneous_expenses,4132170,99.992789
taxes_and_insurance,4132170,99.992789
maintenance_and_preservation_costs,4132170,99.992789
expenses,4132170,99.992789
non_mi_recoveries,4132170,99.992789
delinquent_accrued_interest,4132169,99.992765
net_sales_proceeds,4132169,99.992765


In [15]:
#columns that can be dropped 

drop_columns = [
    "defect_settlement_date",
    "legal_costs",
    "mi_recoveries",
    "miscellaneous_expenses",
    "taxes_and_insurance",
    "maintenance_and_preservation_costs",
    "expenses",
    "non_mi_recoveries",
    "delinquent_accrued_interest",
    "net_sales_proceeds",
    "actual_loss_calculation",
    "modification_cost",
    "current_month_modification_cost", 
    "harp_indicator",
    "pre_harp_loan_sequence_number",
    "super_conforming_flag"
]

merged.drop(columns=drop_columns, inplace=True)

print(merged.columns)

Index(['loan_sequence_number', 'monthly_reporting_period',
       'current_actual_upb', 'current_loan_delinquency_status', 'loan_age',
       'remaining_months_to_legal_maturity', 'modification_flag',
       'zero_balance_code', 'zero_balance_effective_date',
       'current_interest_rate', 'current_deferred_upb',
       'due_date_of_last_paid_installment', 'step_modification_flag',
       'deferred_payment_plan', 'estimated_loan_to_value',
       'zero_balance_removal_upb', 'delinquency_due_to_disaster',
       'borrower_assistance_status_code', 'interest_bearing_upb',
       'credit_score', 'first_payment_date', 'first_time_homebuyer_flag',
       'maturity_date', 'metropolitan_statistical_area',
       'mortgage_insurance_percentage', 'number_of_units', 'occupancy_status',
       'original_combined_loan_to_value', 'original_debt_to_income_ratio',
       'original_unpaid_principal_balance', 'original_loan_to_value',
       'original_interest_rate', 'channel', 'prepayment_penalty_mort

In [17]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    roc_auc_score,
    roc_curve,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

#sample loans for testing - 5000 loans
np.random.seed(42)

unique_loan = merged["loan_sequence_number"].dropna().unique()

sample_size = min(5000, len(unique_loan))

selected_loans = np.random.choice(
    unique_loan,
    size=sample_size,
    replace=False
)

rf_data = merged[
    merged["loan_sequence_number"].isin(selected_loans)
].copy()

rf_data = rf_data.sort_values(
    by=["loan_sequence_number", "monthly_reporting_period"]
)

print("Row: ", len(rf_data))
print("Unique loans: ", rf_data["loan_sequence_number"].nunique())



Row:  412897
Unique loans:  5000


In [ ]:
#convert to binary status 0 or 1
    #1=90+ days delinquent 
    #0=current or fewer than 90 days delinquent
def changeDelinquency(status):
   if pd.isna(status):
      return np.nan
   status = str(status).strip().upper()

   if status == "XX":
      return np.nan
   if status == "R":
      return 1

   try: 
      return 1 if int(float(status)) >= 3 else 0
   except (ValueError, TypeError):
      return np.nan

In [19]:
#current month status
rf_data["changeDelinquency"] = (
    rf_data["current_loan_delinquency_status"]
    .apply(changeDelinquency)
)